In [0]:
passengers_day1 = [
(101,"Rahul Sharma","Hyderabad","Economy","India"),
(102,"Priya Reddy","Bangalore","Business","India"),
(103,"Amit Kumar","Mumbai","Economy","India"),
(104,"Sneha Patel","Delhi","Premium Economy","India"),
(105,"Farhan Ali","Chennai","Economy","India")
]
columns = [
"passenger_id",
"passenger_name",
"city",
"travel_class",
"country"
]
df_day1 = spark.createDataFrame(
passengers_day1,
columns
)

In [0]:
passengers_day2 = [
(102,"Priya Reddy","Bangalore","First Class","India"),
(104,"Sneha Patel","Hyderabad","Premium Economy","India"),
(106,"Neha Singh","Pune","Economy","India"),
(107,"Arjun Verma","Kochi","Business","India")
]

df_day2 = spark.createDataFrame(
passengers_day2,
columns
)

In [0]:
# Save DataFrame as a Delta Table
df_day1.write.format("delta").mode("overwrite").saveAsTable("passengers_delta")

In [0]:

delta_df = spark.read.table("passengers_delta")
print(f"Total records in Day 1: {delta_df.count()}")
delta_df.show()

Total records in Day 1: 5
+------------+--------------+---------+---------------+-------+
|passenger_id|passenger_name|     city|   travel_class|country|
+------------+--------------+---------+---------------+-------+
|         101|  Rahul Sharma|Hyderabad|        Economy|  India|
|         102|   Priya Reddy|Bangalore|       Business|  India|
|         103|    Amit Kumar|   Mumbai|        Economy|  India|
|         104|   Sneha Patel|    Delhi|Premium Economy|  India|
|         105|    Farhan Ali|  Chennai|        Economy|  India|
+------------+--------------+---------+---------------+-------+



In [0]:
from delta.tables import DeltaTable
deltaTable = DeltaTable.forName(spark, "passengers_delta")
deltaTable.history().select("version", "timestamp", "operation", "operationParameters").show(truncate=False)

+-------+-------------------+---------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation                        |operationParameters                                                                                                                                                                                                               |
+-------+-------------------+---------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|0      |2026-06-17 11:46:16|CREATE OR REPLACE TABLE AS SELECT|{isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy 

In [0]:
deltaTable = DeltaTable.forName(spark, "passengers_delta")

deltaTable.alias("target").merge(
    source = df_day2.alias("source"),
    condition = "target.passenger_id = source.passenger_id"
) \
.whenMatchedUpdate(set = {
    "passenger_name": "source.passenger_name",
    "city": "source.city",
    "travel_class": "source.travel_class",
    "country": "source.country"
}) \
.whenNotMatchedInsert(values = {
    "passenger_id": "source.passenger_id",
    "passenger_name": "source.passenger_name",
    "city": "source.city",
    "travel_class": "source.travel_class",
    "country": "source.country"
}) \
.execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:

current_df = spark.read.table("passengers_delta")
current_df.filter("passenger_id = 102").show()
current_df.filter("passenger_id = 106").show()

+------------+--------------+---------+------------+-------+
|passenger_id|passenger_name|     city|travel_class|country|
+------------+--------------+---------+------------+-------+
|         102|   Priya Reddy|Bangalore| First Class|  India|
+------------+--------------+---------+------------+-------+

+------------+--------------+----+------------+-------+
|passenger_id|passenger_name|city|travel_class|country|
+------------+--------------+----+------------+-------+
|         106|    Neha Singh|Pune|     Economy|  India|
+------------+--------------+----+------------+-------+



In [0]:

df_version_0 = spark.read.format("delta").option("versionAsOf", 0).table("passengers_delta")
df_latest = spark.read.table("passengers_delta")

In [0]:
print(f"Version 0 Count: {df_version_0.count()}") # Expected: 5
print(f"Latest Version Count: {df_latest.count()}") # Expected: 7 (5 original - 0 deleted + 2 new)

Version 0 Count: 5
Latest Version Count: 7


In [0]:
print("Passenger 102 in Version 0:")
df_version_0.filter("passenger_id = 102").select("passenger_id", "travel_class").show()

print("Passenger 102 in Latest Version:")
df_latest.filter("passenger_id = 102").select("passenger_id", "travel_class").show()

Passenger 102 in Version 0:
+------------+------------+
|passenger_id|travel_class|
+------------+------------+
|         102|    Business|
+------------+------------+

Passenger 102 in Latest Version:
+------------+------------+
|passenger_id|travel_class|
+------------+------------+
|         102| First Class|
+------------+------------+



In [0]:
print("Passenger 104 in Version 0:")
df_version_0.filter("passenger_id = 104").select("passenger_id", "city").show()

print("Passenger 104 in Latest Version:")
df_latest.filter("passenger_id = 104").select("passenger_id", "city").show()

Passenger 104 in Version 0:
+------------+-----+
|passenger_id| city|
+------------+-----+
|         104|Delhi|
+------------+-----+

Passenger 104 in Latest Version:
+------------+---------+
|passenger_id|     city|
+------------+---------+
|         104|Hyderabad|
+------------+---------+



In [0]:
# You can execute optimization configurations directly on the Delta object
deltaTable.optimize().executeZOrderBy("city")

DataFrame[path: string, metrics: struct<autoCompactParallelismStats:void,clusteringMetrics:void,clusteringStats:void,deletionVectorStats:struct<numDeletionVectorRowsRemoved:bigint,numDeletionVectorsRemoved:bigint>,endTimeMs:bigint,fileAgeHistogram:void,filesAdded:struct<avg:double,max:void,min:void,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<avg:double,max:void,min:void,totalFiles:bigint,totalSize:bigint>,numBatches:bigint,numBins:bigint,numBytesSkippedToReduceWriteAmplification:bigint,numFilesAdded:bigint,numFilesRemoved:bigint,numFilesSkippedToReduceWriteAmplification:bigint,numTableColumns:bigint,numTableColumnsWithStats:bigint,partitionsOptimized:bigint,preserveInsertionOrder:boolean,recompressionCodec:void,skippedArchivedFiles:bigint,startTimeMs:bigint,totalClusterParallelism:bigint,totalConsideredFiles:bigint,totalFilesSkipped:bigint,totalScheduledTasks:bigint,totalTaskExecutionTimeMs:bigint,zOrderStats:struct<inputCubeFiles:struct<num:bigint,size:bigint>,inputNumCube

In [0]:
deltaTable.delete(condition = "passenger_id = 105")

DataFrame[num_affected_rows: bigint]

In [0]:

deltaTable.history().select("version", "operation", "operationMetrics").show(3, truncate=False)

+-------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|operation|operationMetrics                                                                                                                                                                                                                                                                                                        |
+-------+---------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|

In [0]:


# Remove older files no longer required by the latest table state snapshot
# Note: On Serverless compute, the retention check cannot be disabled.
# Minimum retention is 168 hours (7 days) to protect against concurrent writes.
deltaTable.vacuum(168)